For licensing see accompanying LICENSE file.  
Copyright (C) 2025 Apple Inc. All Rights Reserved.

# Evaluation of the Human Study Data

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent
os.chdir(ROOT) # Change working directory to project root
sys.path.insert(0, str(ROOT))

In [3]:
import json
import numpy as np
import pandas as pd
import altair as alt
from tqdm import tqdm

import features

## Load Human Study Responses

In [4]:
# survey to feature map
model_id = 'gpt2-small'
survey_to_feature_map = [
    [
        {"feature": {"model_id": model_id, "layer": '1-res-jb',  "index": 7218 }, 'description_type': 'sr', 'feature_type': 'symbol'},
        {"feature": {"model_id": model_id, "layer": '10-res-jb', "index": 737  }, 'description_type': 'sr', 'feature_type': 'field'},
        {"feature": {"model_id": model_id, "layer": '3-res-jb',  "index": 19219}, 'description_type': 'sr', 'feature_type': 'context'},
        {"feature": {"model_id": model_id, "layer": '6-res-jb',  "index": 11289}, 'description_type': 'nl', 'feature_type': 'lexeme'},
        {"feature": {"model_id": model_id, "layer": '7-res-jb',  "index": 9878 }, 'description_type': 'nl', 'feature_type': 'combo'},
        {"feature": {"model_id": model_id, "layer": '10-res-jb', "index": 5144 }, 'description_type': 'nl', 'feature_type': 'combo'},
    ],
    [
        {"feature": {"model_id": model_id, "layer": '1-res-jb',  "index": 7218 }, 'description_type': 'nl', 'feature_type': 'symbol'},
        {"feature": {"model_id": model_id, "layer": '10-res-jb', "index": 737  }, 'description_type': 'nl', 'feature_type': 'field'},
        {"feature": {"model_id": model_id, "layer": '3-res-jb',  "index": 19219}, 'description_type': 'nl', 'feature_type': 'context'},
        {"feature": {"model_id": model_id, "layer": '6-res-jb',  "index": 11289}, 'description_type': 'sr', 'feature_type': 'lexeme'},
        {"feature": {"model_id": model_id, "layer": '7-res-jb',  "index": 9878 }, 'description_type': 'sr', 'feature_type': 'combo'},
        {"feature": {"model_id": model_id, "layer": '10-res-jb', "index": 5144 }, 'description_type': 'sr', 'feature_type': 'combo'},
    ],
    [
        {"feature": {"model_id": model_id, "layer": '1-res-jb',  "index": 4484 }, 'description_type': 'sr', 'feature_type': 'symbol'},
        {"feature": {"model_id": model_id, "layer": '8-res-jb',  "index": 21194}, 'description_type': 'sr', 'feature_type': 'field'},
        {"feature": {"model_id": model_id, "layer": '9-res-jb',  "index": 14593}, 'description_type': 'sr', 'feature_type': 'context'},
        {"feature": {"model_id": model_id, "layer": '8-res-jb',  "index": 21078}, 'description_type': 'nl', 'feature_type': 'field'},
        {"feature": {"model_id": model_id, "layer": '4-res-jb',  "index": 4484 }, 'description_type': 'nl', 'feature_type': 'combo'},
        {"feature": {"model_id": model_id, "layer": '1-res-jb',  "index": 19894}, 'description_type': 'nl', 'feature_type': 'combo'},
    ],
    [
        {"feature": {"model_id": model_id, "layer": '1-res-jb',  "index": 4484 }, 'description_type': 'nl', 'feature_type': 'symbol'},
        {"feature": {"model_id": model_id, "layer": '8-res-jb',  "index": 21194}, 'description_type': 'nl', 'feature_type': 'field'},
        {"feature": {"model_id": model_id, "layer": '9-res-jb',  "index": 14593}, 'description_type': 'nl', 'feature_type': 'context'},
        {"feature": {"model_id": model_id, "layer": '8-res-jb',  "index": 21078}, 'description_type': 'sr', 'feature_type': 'field'},
        {"feature": {"model_id": model_id, "layer": '4-res-jb',  "index": 4484 }, 'description_type': 'sr', 'feature_type': 'combo'},
        {"feature": {"model_id": model_id, "layer": '1-res-jb',  "index": 19894}, 'description_type': 'sr', 'feature_type': 'combo'},
    ],

]

In [5]:
def split_survey_responses(response_string):
    """Splits participant positive generations into a list of three phrases."""
    response_string = response_string.strip()
    split_newline = response_string.split('\n')
    if len(split_newline) == 3:
        return split_newline

    split_comma = response_string.split(',')
    if len(split_comma) == 3:
        return [s.strip() for s in split_comma]

    split_semicolon = response_string.split(';')
    if len(split_semicolon) == 3:
        return [s.strip() for s in split_semicolon]

    split_period = response_string.split('. ')
    if len(split_period) == 3:
        return [s.strip() for s in split_period]

    print(f"Could not split response string: {response_string}")
    return [response_string]


def strip_survey_responses(responses):
    """Removes common prefixes from survey responses."""
    if np.all([r.startswith(f'{i+1}. ') for i, r in enumerate(responses)]):
        return [r[3:].strip() for r in responses]

    if np.all([r.startswith('- ') for r in responses]):
        return [r[2:].strip() for r in responses]

    return responses


In [6]:
def load_survey_features(survey_feature_dir):
    """Get the features and their descriptions from experiments/human_study/"""
    sr_human_study_feature_dir = os.path.join(survey_feature_dir, 'cf1daa_semantic_regex_gpt2-small_res-jb')
    nl_human_study_feature_dir = os.path.join(survey_feature_dir, 'cf1daa_eleuther_acts_top20_gpt2-small_res-jb')


    human_study_features = {}
    for feature_file in os.listdir(sr_human_study_feature_dir):
        if feature_file.endswith('.json'):
            with open(os.path.join(sr_human_study_feature_dir, feature_file), 'r') as f:
                sr_experiment = json.load(f)
                sr_feature = features.Feature(**sr_experiment['feature'])
                sr_description = sr_experiment['description']['description']
            with open(os.path.join(nl_human_study_feature_dir, feature_file), 'r') as f:
                nl_experiment = json.load(f)
                nl_feature = features.Feature(**nl_experiment['feature'])
                nl_description = nl_experiment['description']['description']
            assert str(sr_feature) == str(nl_feature)
            human_study_features[str(sr_feature)] = {
                'feature': sr_feature.to_dict(),
                'sr_description': sr_description,
                'nl_description': nl_description,
                'sr_responses': [],
                'nl_responses': [],
            }

    return human_study_features

In [7]:
def parse_survey_responses(survey_response_dir, feature_responses):
    """Get the features and their descriptions from experiments/human_study/"""
    # parse human study results

    num_surveys = len(survey_to_feature_map)

    for survey_index in range(num_surveys):
        with open(os.path.join(survey_response_dir, f'survey_{survey_index+1}_responses.json'), 'r') as f:
            survey_responses = json.load(f)

        for participant in survey_responses:
            participant_id = participant['participant_id']
            responses = participant['responses']
            num_questions = len(responses) // 2
            for question_index in range(num_questions):
                question = survey_to_feature_map[survey_index][question_index]
                question_feature = features.Feature(**question['feature'])
                question_type = question['description_type']

                match_response = responses[2 * question_index]
                counterfactual_response = responses[2 * question_index + 1]

                human_study_feature = feature_responses[str(question_feature)]
                assert human_study_feature[f'{question_type}_description'][:20] in match_response['label'], f"{human_study_feature[f'{question_type}_description']} not in {match_response['label']}"
                assert human_study_feature[f'{question_type}_description'][:20] in counterfactual_response['label'], f"{human_study_feature[f'{question_type}_description']} not in {counterfactual_response['label']}"

                response_record = {'pid': participant_id, 'positives': [], 'counterfactuals': []}
                # generate a match response
                match_generations = match_response['response']
                assert len(match_generations) == 1
                split_match_generations = strip_survey_responses(split_survey_responses(match_generations[0]))
                if 'n/a' in [s.lower() for s in split_match_generations]:
                    print(f"Skipping response with n/a: {split_match_generations}")
                    continue


                # generate a counterfactual response
                counterfactual = counterfactual_response['response']
                assert len(counterfactual) == 1
                if 'n/a' in [s.lower() for s in counterfactual]:
                    print(f"Skipping response with n/a: {counterfactual}")
                    continue

                response_record['positives'] = split_match_generations
                response_record['counterfactuals'] = [counterfactual[0]]

                feature_responses[str(question_feature)][f'{question_type}_responses'].append(response_record)

    return feature_responses



In [8]:
human_study_feature_dir = os.path.join(ROOT, 'artifacts', 'experiments', 'human_study')
human_study_results_dir = os.path.join(ROOT, 'human_study', 'responses')
response_file = os.path.join('human_study', 'human_study_responses.json')

if os.path.exists(response_file):
    with open(response_file, 'r') as f:
        human_study_features = json.load(f)
else:
    human_study_features = load_survey_features(human_study_feature_dir)
    human_study_features = parse_survey_responses(human_study_results_dir, human_study_features)
    with open(response_file, 'w') as f:
        json.dump(human_study_features, f, indent=2)

## Compute Feature Activations

In [9]:
def compute_response_activations(human_study_features):
    """Compute feature activations on the survery responses."""
    for feature_name, results in tqdm(human_study_features.items()):
        feature = features.Feature(**results['feature'])
        for sr_response in results['sr_responses']:
            positives = sr_response['positives']
            sr_response['positive_tokens'] = []
            sr_response['positive_activations'] = []
            for positive in positives:
                positive  = ' ' + positive  # add a space to the beginning to account for tokenizer behavior
                tokens, activations = feature.query_feature(positive, ignore_first_token=True)
                sr_response['positive_tokens'].append(tokens)
                sr_response['positive_activations'].append(activations)

            counterfactuals = sr_response['counterfactuals']
            sr_response['counterfactual_tokens'] = []
            sr_response['counterfactual_activations'] = []
            for counterfactual in counterfactuals:
                counterfactual = ' ' + counterfactual  # add a space to the beginning to account for tokenizer behavior
                tokens, activations = feature.query_feature(counterfactual, ignore_first_token=True)
                sr_response['counterfactual_tokens'].append(tokens)
                sr_response['counterfactual_activations'].append(activations)

        for nl_response in results['nl_responses']:
            positives = nl_response['positives']
            nl_response['positive_tokens'] = []
            nl_response['positive_activations'] = []
            for positive in positives:
                positive  = ' ' + positive  # add a space to the beginning to account for tokenizer behavior
                tokens, activations = feature.query_feature(positive, ignore_first_token=True)
                nl_response['positive_tokens'].append(tokens)
                nl_response['positive_activations'].append(activations)

            counterfactuals = nl_response['counterfactuals']
            nl_response['counterfactual_tokens'] = []
            nl_response['counterfactual_activations'] = []
            for counterfactual in counterfactuals:
                counterfactual = ' ' + counterfactual  # add a space to the beginning to account for tokenizer behavior
                tokens, activations = feature.query_feature(counterfactual, ignore_first_token=True)
                nl_response['counterfactual_tokens'].append(tokens)
                nl_response['counterfactual_activations'].append(activations)
    return human_study_features



In [10]:
human_study_results_file = os.path.join('human_study', 'human_study_results.json')

if os.path.exists(human_study_results_file):
    with open(human_study_results_file, 'r') as f:
        human_study_results = json.load(f)
else:
    human_study_results = compute_response_activations(human_study_features)
    with open(human_study_results_file, 'w') as f:
        json.dump(human_study_results, f, indent=2)

## Analyze the Results

In [11]:
def human_study_json_to_df(human_study_results):
    """Convert the human study results JSON to a pandas DataFrame."""
    rows = []

    for feature_str, item in human_study_results.items():

        # Iterate over description types (sr and nl)
        for desc_type, desc_key in [("sr", "sr_description"), ("nl", "nl_description")]:
            description = item.get(desc_key, "")
            responses_key = f"{desc_type}_responses"

            for response in item.get(responses_key, []):
                pid = response["pid"]

                # Handle positives
                for resp, activations in zip(
                    response.get("positives", []),
                    response.get("positive_activations", [])
                ):
                    rows.append({
                        "feature": feature_str,
                        "description": description,
                        "description type": desc_type,
                        "participant": pid,
                        "response": resp,
                        "max activation": max(activations) if activations else None,
                        "response type": "positive"
                    })

                # Handle counterfactuals
                for resp, activations in zip(
                    response.get("counterfactuals", []),
                    response.get("counterfactual_activations", [])
                ):
                    rows.append({
                        "feature": feature_str,
                        "description": description,
                        "description type": desc_type,
                        "participant": pid,
                        "response": resp,
                        "max activation": max(activations) if activations else None,
                        "response type": "counterfactual"
                    })

    return pd.DataFrame(rows)

df = human_study_json_to_df(human_study_results)
df.head()

,feature,description,description type,participant,response,max activation,response type
0,gpt2-small_1-res-jb_7218,[:symbol on:],sr,d78954b6-018b-4d56-9cf5-13faaf73d689,My hat is on my dog,26.721899,positive
1,gpt2-small_1-res-jb_7218,[:symbol on:],sr,d78954b6-018b-4d56-9cf5-13faaf73d689,water bottle on a desk,23.005060,positive
2,gpt2-small_1-res-jb_7218,[:symbol on:],sr,d78954b6-018b-4d56-9cf5-13faaf73d689,getting on a flight,22.287642,positive
3,gpt2-small_1-res-jb_7218,[:symbol on:],sr,d78954b6-018b-4d56-9cf5-13faaf73d689,My dog has my hat,0.000000,counterfactual
4,gpt2-small_1-res-jb_7218,[:symbol on:],sr,dc4f6151-38ff-43d4-ae34-93364c8b4fe4,The lock is active.,0.000000,positive


In [12]:
# Add max feature activation to the dataframe
max_activations = {}
for feature_str, item in human_study_results.items():
    feature = features.Feature(**item['feature'])
    feature_info = feature.get_feature_info()
    max_activations[feature_str] = feature_info['maxActApprox']

df['max_feature_activation'] = df['feature'].map(max_activations)

# Swap description types for their actual method names
df['description type'] = df['description type'].replace({'nl': 'max-acts', 'sr': 'semantic-regex'})

# Add the feature type to the dataframe
feature_types = {}
for survey in survey_to_feature_map:
    for question in survey:
        feature = features.Feature(**question['feature'])
        feature_types[str(feature)] = question['feature_type']
df['feature type'] = df['feature'].map(feature_types)

In [16]:
def print_feature_data(df):
    """Print the survey results in a readable format."""
    # Loop over features
    for feat, feat_group in df.groupby("feature"):
        description = feat_group["description"].unique()
        print("=" * 100)
        print(f"Feature: {feat}")
        print(f"Description: {description}\n")

        # Figure out whether a participant is NL or SR (based on any of their rows)
        participant_types = (
            feat_group.groupby("participant")["description type"]
            .first()
            .map({"max-acts": 0, "semantic-regex": 1})
        )

        # Reorder participants: NL first, then SR
        for pid in participant_types.sort_values().index:
            part_group = feat_group[feat_group["participant"] == pid]
            desc_type = part_group["description type"].iloc[0]

            print(f"  Participant: {pid}  ({desc_type.upper()})")
            for _, row in part_group.iterrows():
                print(f"    - {row['max activation']:.2f} | "
                      f"({row['response type']}) | "
                      f"{row['response']} "
                      )
            print()

print_feature_data(df)


Feature: gpt2-small_1-res-jb_19894
Description: ['@{:context academic citations}([:symbol DOI:] | [:field reference:])'
 'Frequent use of citation formats, including publication years, journal names, and DOI references, indicating scholarly writing.']

  Participant: 0c4d47ce-12a6-4e10-b848-6f455b22afa1  (MAX-ACTS)
    - 0.00 | (positive) | The paper "Attention is All You Need" was published in 2017. 
    - 0.00 | (counterfactual) | Is life like a box of chocolates? 

  Participant: 10d42194-fc0a-46f2-9889-b2169cb886ca  (MAX-ACTS)
    - 6.76 | (positive) | Maryam Taeb, Amanda Swearngin, Eldon Schoop, Ruijia Cheng, Yue Jiang, and Jeffrey Nichols. 2024. AXNav: Replaying Accessibility Tests from Natural Language. In Proceedings of the 2024 CHI Conference on Human Factors in Computing Systems (CHI '24). Association for Computing Machinery, New York, NY, USA, Article 962, 1–16. https://doi.org/10.1145/3613904.3642777 
    - 0.00 | (positive) | Cite the paper in Harvard style 
    - 6.88 | (

In [15]:
def get_feature_name(feature_str, feature_type):
    """Process feature name for plotting."""
    _, layer, index = feature_str.split('_')
    layer = layer.split('-')[0]
    return f"res-{layer} {index}"

def save_chart(chart, filename):
    """Save an Altair chart with higher resolution."""
    chart.save(
        filename,
        scale_factor=2,
        format='png',
        ppi=300,
    )

def plot_dumbbell(df):

    # get max positive response per feature-participant
    pos_df = (
        df[df["response type"] == "positive"]
        .groupby(["feature", "participant", "description type", "description", "max_feature_activation", "feature type"], as_index=False)["max activation"]
        .max()
        .rename(columns={"max activation": "max_positive_activation"})
    )

    # get counterfactual activation per feature-participant
    cf_df = (
        df[df["response type"] == "counterfactual"]
        .groupby(["feature", "participant"], as_index=False)["max activation"]
        .first()
        .rename(columns={"max activation": "counterfactual_activation"})
    )

    # merge them
    merged = pd.merge(pos_df, cf_df, on=["feature", "participant"], how="inner")

    # compute difference
    merged["diff"] = merged["max_positive_activation"] - merged["counterfactual_activation"]

    # get mean diff per feature-description type
    means = (
        merged
        .groupby(["feature", "description type", "feature type", "max_feature_activation"], as_index=False)
        .agg(mean_diff=("diff", "mean"), n=("diff", "count"))
    )
    means['mean_diff_percentage'] = means['mean_diff'] / means['max_feature_activation']
    means['feature name'] = means.apply(lambda row: get_feature_name(row['feature'], row['feature type']), axis=1)



    # Add logic to determine which endpoint is higher per feature
    def determine_line_color(group):
        if len(group) >= 2:
            values = group['mean_diff_percentage'].values
            return 'blue' if values.max() == values[0] else 'red'
        return 'gray'  # fallback for single points

    # Group by feature and determine color based on which description type has higher value
    color_mapping = (
        means.groupby('feature name')
        .apply(determine_line_color)
        .reset_index()
        .rename(columns={0: 'line_color'})
    )

    # Merge back to means
    means = means.merge(color_mapping, on='feature name', how='left')




    # order features by sr value
    sr_mask = means["description type"].str.lower().isin(["semantic-regex"])
    order = (
        means.loc[sr_mask]
        .groupby("feature name", as_index=False)["mean_diff_percentage"].mean()  # in case of duplicates
        .sort_values("mean_diff_percentage", ascending=True)                # flip to False for descending
        ["feature name"].tolist()
    )

    # line connecting NL→SR within feature
    lines = (
        alt.Chart(means)
        .mark_line(strokeWidth=2)
        .encode(
            y=alt.Y("feature name:N", sort=order, title="LLM Feature").axis(
            titleAngle=0,  # Make horizontal
            titleAnchor='middle',
            titleY=-10,  # Position above the labels
            titleX=-45,   # Adjust horizontal position as needed
        ),
            x=alt.X("min(mean_diff_percentage):Q", title=["Max Diff. of User Positive and Counterfactual Activations","(mean per feature; % of feature's max data activation)"]).axis(format='%'),
            x2="max(mean_diff_percentage):Q",
            # color=alt.value('lightgray'),
        color=alt.Color("line_color:N", scale=alt.Scale(domain=['red', 'blue', 'gray'], range=['#d62728', '#17becf', 'gray']), legend=None),
        )
    )

    # points for NL & SR means (size ~ n)
    points = (
        alt.Chart(means)
        .mark_circle(size=100, opacity=1)
        .encode(
            y=alt.Y("feature name:N", sort=order),
            x=alt.X("mean_diff_percentage:Q"),
            color=alt.Color("description type:N", title="Type", scale=alt.Scale(range=['#17becf', '#d62728'])),
        )
    )

    chart_dumbbell = alt.layer(lines, points).resolve_scale(color='independent').properties(
        # width=520,
        # height=18 * max(1, len(order)),
        title="Human Generatated Feature Activations"
    ).configure_title(
        anchor='middle',
        fontSize=18,
        dy=-5
    ).configure_axis(
        titleFontSize=12,
        titleColor="grey",
    ).configure_legend(
        labelFontSize=14,
        titleFontSize=14,
        orient='none',
        legendX=40,
        legendY=-25,
        direction='horizontal',
        titleBaseline='middle',
        titleOrient='left',
    )

    return chart_dumbbell

db = plot_dumbbell(df)
save_chart(db, os.path.join('human_study', 'human_study_dumbbell.png'))
db

/var/folders/c6/8tprqv8s24l28zvgqgj4__nw0000gn/T/ipykernel_13041/859992752.py:53: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(determine_line_color)


alt.LayerChart(...)